In [42]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

True

In [43]:
# Initialize Anthropic client with beta header for PDF support
client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    default_headers={
        "anthropic-beta": "pdfs-2024-09-25"  # Required for PDF support
    }
)

## Basic PDF Analyser using API
ref: https://docs.anthropic.com/en/docs/build-with-claude/pdf-support

In [46]:
def analyze_pdf(file_path, prompt):
    with open(file_path, "rb") as f:
        # Convert bytes to base64 string
        import base64
        file_data = base64.b64encode(f.read()).decode('utf-8')

        message = client.beta.messages.create(
            model="claude-3-5-sonnet-20241022",
            betas=["pdfs-2024-09-25"],
            max_tokens=1024,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": file_data
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }]
        )
        return message.content

In [47]:
# Example usage
file_path = "Memo 3.pdf"
prompt = "What is the summary of the document?"
print(analyze_pdf(file_path, prompt))

[BetaTextBlock(text="This document is an internal memo from Alliance Bank dated 11 Oct 2024 requesting approval to streamline Business Current Account opening processes. Here are the key points:\n\n1. Purpose:\n- To standardize account opening document requirements for various entities like Escrow, Joint Management Body, Religious Body, Government School, Non-Government School/Kindergarten, and Parent-Teacher Association\n- To refine and clarify business current account opening requirements and processes\n\n2. Background:\n- A revised account opening checklist was implemented on 1 August 2024 but some entities were not included\n- There's a need to refine supporting documents and validity periods to reduce ambiguity\n- Special considerations needed for Sabah companies due to different documentation requirements\n- Some processes need updates and refinement\n\n3. Benefits:\n- Improve account opening turnaround time\n- Simplify customer journey\n- Minimize rework and audit findings\n\n4.

## Multiple pdfs analysis

### Limitations 
- Maximum pages per request: 100 pages 
- Maximum request size:	32MB
- Supported models: claude-3-5-sonnet-20241022, claude-3-5-sonnet-20240620

In [48]:
def analyze_multiple_pdfs(file_paths, prompt):
    # Prepare all PDF documents
    documents = []
    for file_path in file_paths:
        with open(file_path, "rb") as f:
            import base64

            file_data = base64.b64encode(f.read()).decode('utf-8')
            documents.append({
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": file_data
                },
            })

    # Add the prompt as the final text element
    documents.append({
        "type": "text",
        "text": prompt
    })

    # Send all documents in a single request
    message = client.beta.messages.create(
        model="claude-3-5-sonnet-20241022",
        betas=["pdfs-2024-09-25"],
        # 32mb
        max_tokens=4096,
        messages=[{
            "role": "user",
            "content": documents
        }]
    )

    return message.content

# Example usage:
files = ["MN1005.pdf"]
prompt = "Compare and contrast the key points from all three documents."
result = analyze_multiple_pdfs(files, prompt)
print(result)

[BetaTextBlock(text='These three documents appear to be different versions or parts of a Standard Operating Procedures (SOP) manual for Alliance Bank\'s Conventional Business Deposit Product. Here are the key points of comparison and contrast:\n\nCommon Elements Across Documents:\n1. All are dated June 30, 2024 with a next review date of June 30, 2026\n2. Version number 1.1/2024 is consistent across all documents\n3. Share the same basic document structure and formatting\n4. All are marked "For Internal Use Only"\n\nKey Differences:\n\nDocument 1 (Pages 1-25):\n- Contains introductory sections and basic definitions\n- Focuses on account opening procedures and documentation requirements\n- Includes general policies and administrative details\n\nDocument 2 (Pages 26-50):\n- More detailed operational procedures\n- Specific workflows for different types of accounts\n- Contains technical requirements and system procedures\n\nDocument 3 (Pages 51-92):\n- Foreign exchange rules and regulation